# 06 · The Voigt function and line profiles

Every gas element absorbs a transition over a range of frequencies, its line profile. Two effects set its width: the
motions of the absorbing ions, thermal and turbulent, which broaden it into a Gaussian (the Doppler width $b$), and
the finite lifetime of the upper level, natural and collisional, which gives it Lorentzian wings. Together they make a
Voigt profile, the shape that every piece of gas contributes to the optical depth.

This notebook checks the Voigt function against its limits and a reference implementation, with a fast
approximation (§1), derives the profile of a gas element from its temperature, velocity and scale factor (§2,
`core.line_profiles`), and compares the exact detuning with its first order (§3). Notebook 07 cuts the skewers into
the pieces of the gas cells and sums these profiles into $\tau$. Runtime: ~15 s.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag)

In [ ]:
plot_dag(('lines', 'profiles',)); plt.show()

In [ ]:
from skewerjax.core import LineBundle, LINES, voigt as V, phys
from skewerjax.core.line_profiles import line_profiles, voigt_widths_GHz
from skewerjax.core.spectral import line_centre_velocity_kms
lines = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"], ref_line_id="HI-LyA")
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")

## 1 · The Voigt function

Three implementations of the same function $V(x; \sigma, \gamma) = (G_\sigma * L_\gamma)(x)$, the convolution of a
Gaussian of standard deviation $\sigma$ and a Lorentzian of half-width $\gamma$:

- `"ExoJAX"` (default): a Humlíček-type approximation of the Faddeeva function, compatible with `jit` and `grad`;
- `"convolution"`: $G * L$ by FFT on the grid, refined 4 times; it needs a grid that resolves both widths;
- `"harris"`: the first-order expansion in the damping parameter $a = \gamma/(\sqrt2\, \sigma)$ (Harris 1948), the
  fast scheme of other codes, below; $\mathrm{Ly}\alpha$ has $a \approx 5\times10^{-4}$ at $10^4$ K;
- `"scipy_wofz"`: the reference, in NumPy, with

$$
V = \frac{\mathrm{Re}\, w(z)}{\sqrt{2\pi}\, \sigma},\qquad z = \frac{x + i\gamma}{\sqrt2\, \sigma} .
$$

The four $(\sigma, \gamma)$ pairs combine $10^{-3}$ and $1$:

- the Gaussian limit ($\gamma \ll \sigma$);
- the Lorentzian limit ($\sigma \ll \gamma$);
- two Voigt profiles with both widths equal.

In [ ]:
from skewerjax.utils.plotting import plot_voigt_limits
x = jnp.linspace(-9, 9, 2 ** 13)
sigma = np.array([1e-3, 1.0, 1e-3, 1.0]); gamma = np.array([1e-3, 1e-3, 1.0, 1.0])
curves = {"scipy_wofz": np.stack([V.voigt_scipy(np.asarray(x), s, g) for s, g in zip(sigma, gamma)]),
          "ExoJAX": np.stack([np.asarray(V.voigt_exojax(x, s, g)) for s, g in zip(sigma, gamma)]),
          "convolution (FFT, oversample 4)": np.stack([np.asarray(V.voigt_convolution(x, s, g, oversample=4)) for s, g in zip(sigma, gamma)]),
          "harris (first order in a)": np.stack([np.asarray(V.voigt_harris(x, s, g)) for s, g in zip(sigma, gamma)])}
# the expansion alone (no exact fallback) against wofz, over the damping parameters where it is used: H = sqrt(pi) V(u; 1/sqrt2, a)
u = np.geomspace(1e-2, 40.0, 4000)
expansion = {a: np.abs(np.asarray(V.voigt_harris(jnp.asarray(u), 2 ** -0.5, a, a_max=None)) / V.voigt_scipy(u, 2 ** -0.5, a) - 1)
             for a in (1e-5, 1e-4, 1e-3)}                         # |H / H_exact - 1|, printed below
fig, axs = plot_voigt_limits(np.asarray(x), sigma, gamma, curves)
add_panel_letters(axs)
fig.suptitle("Voigt: every implementation on scipy's wofz, from the Gaussian to the Lorentzian limit", fontsize=21)
plt.show()
for name in ("ExoJAX", "convolution (FFT, oversample 4)", "harris (first order in a)"):
    r = np.abs(curves[name] / curves["scipy_wofz"] - 1)
    core = np.abs(np.asarray(x))[None] < 3 * np.maximum(sigma, gamma)[:, None]
    print(f"{name:32s} max |V / wofz - 1| per pair (|x| < 3 max(sigma, gamma)):", np.array2string(np.where(core, r, 0).max(1), precision=1))
print("harris, a = gamma / (sqrt2 sigma) per pair:", np.array2string(gamma / (np.sqrt(2) * sigma), precision=1),
      f"-> the expansion for a <= {V.HARRIS_A_MAX:g}, the exact function above")
for a, r in expansion.items():
    print(f"harris alone, a = {a:g}: max |H / H_exact - 1| core (u < 3) {r[u < 3].max():.1e}, transition (3-6) "
          f"{r[(u >= 3) & (u < 6)].max():.1e}, wings (u >= 6) {r[u >= 6].max():.1e}")
sig_ly, gam_ly = (np.asarray(w)[:, 0] for w in voigt_widths_GHz(jnp.full((3, 1), 1e4), jnp.zeros((3, 1)), jnp.zeros((3, 1)), jnp.asarray(lines.lambda0_A),
                                                                  jnp.asarray(lines.A21_GHz), jnp.asarray(lines.mass_amu)))
print("a at T = 1e4 K:", ", ".join(f"{l.id} {g / (np.sqrt(2) * s):.1e}" for l, s, g in zip(lines.lines, sig_ly, gam_ly)))

ExoJAX agrees with `wofz` to $\sim 10^{-6}$ or better for every pair. The convolution is exact ($10^{-14}$) when the
grid resolves both widths, and it fails where the grid cannot:

- $\sigma = \gamma = 10^{-3}$ is far below the grid spacing ($2.2\times10^{-3}$);
- the Lorentzian wings are cut at the grid ends.

The first-order expansion, $H(a, u) \simeq e^{-u^2} + a\, H_1(u)$ with $H_1 = 2u\, \mathrm{Im}\, w(u) - 2/\sqrt\pi$
tabulated and an asymptotic series beyond $u = 6$, replaces the Faddeeva function where $a \le 10^{-3}$ and falls back to
the exact one above (here, three of the four pairs). Its error is the neglected $a^2 (1 - 2u^2) e^{-u^2}$ in the core,
largest ($10^{-5}$ at $a = 10^{-3}$) where the Gaussian core hands over to the Lorentzian wings, $u \approx 3$, and the
truncated wing series in the wings, $6\times10^{-6}$ at $u = 6$ and falling as $u^{-10}$ (printed above, for the
expansion alone at $a = 10^{-5}$, $10^{-4}$ and $10^{-3}$): far below any
effect on $\tau$ for $\mathrm{Ly}\alpha$ and the Lyman series, while the metal lines, with larger $a$, take the exact path.

Full checks: `scripts/validation/check_voigt.py`.

## 2 · Line profiles of gas elements

A gas element at $x$ absorbs a photon of transition $j$ according to how far the photon is from the line in the gas's
own frame. Four steps give that distance and the profile.

**The profile.** It is the Voigt function of §1 at the frequency detuning $\nu_j$ of the photon (review notes,
Eq. `line_profile`),

$$
\phi_j = V\big(\nu_j;\, \sigma, \gamma\big),\qquad \int V\, d\nu = 1 ,
$$

with unit area in $\nu$ [GHz], so $\phi$ is in ns. Its widths come from the gas and the transition:

- $\sigma = b/(\sqrt2\, \lambda_j)$, the Gaussian width, with $b = \sqrt{2k_BT/m} + b_{\rm turb}$: the thermal motions of
  the ions of mass $m$ at temperature $T$, plus unresolved turbulence;
- $\gamma = (A_{21} + A_{\rm coll})/(4\pi)$, the Lorentzian half-width: the natural decay rate of the upper level plus
  collisions.

The package adds $b_{\rm turb}$ linearly; the review notes add it in quadrature. Both agree for $b_{\rm turb} = 0$, the
default.

**The spectral coordinate.** The observer labels every photon by its line-centred velocity $v_\dagger$, a relabelling of
the observed wavelength, $\lambda_{\rm obs} = \lambda_j\, (a_{\rm obs}/a_\dagger)\, e^{v_\dagger/c}$, with a reference scale
factor $a_\dagger$.

**The local wavelength.** At $x$, a gas element with scale factor $a(x)$ and velocity $v_\parallel$ along the photon sees it
at

$$
\lambda'_j = \lambda_j\, e^{v_\dagger/c}\, \frac{a(x)}{a_\dagger}\, D^{-1}(x),
\qquad D^{-1} = \sqrt{\frac{1+\beta}{1-\beta}},\qquad \beta = \frac{v_\parallel}{c}
$$

(review notes, Eq. `velocity_units_normalized`). Each factor has one role:

- $e^{v_\dagger/c}$ is the photon's label;
- $a(x)/a_\dagger$ undoes the cosmological redshift between $x$ and the reference;
- $D^{-1}$ is the relativistic Doppler factor of the gas moving along the photon.

**The line centre.** The element absorbs at line centre ($\lambda'_j = \lambda_j$) when $v_\dagger$ equals

$$
v_c(x) = c\left[\ln\frac{a_\dagger}{a(x)} + \frac12 \ln\frac{1-\beta}{1+\beta}\right] ,
$$

the sum of a Hubble part, $v_H = c\ln(a_\dagger/a)$, and a peculiar part, $-c\, \mathrm{artanh}\, \beta \simeq -v_\parallel$.

**The detuning.** It is then an exact function of $v_\dagger - v_c$:

$$
\nu_j = c\left(\frac{1}{\lambda'_j} - \frac{1}{\lambda_j}\right) = \frac{c}{\lambda_j}\left[e^{-(v_\dagger - v_c)/c} - 1\right] .
$$

The profile of a gas element is therefore the same function of $v_\dagger - v_c$ wherever the element is
(`core.spectral.frequency_offset_GHz`). $T$, $b_{\rm turb}$ and $A_{\rm coll}$ set its shape; $a$ and $v_\parallel$ set
only its centre.

`line_profiles(v, lines, temperature=..., los_velocity=..., scale_factor=..., a_dagger=...)` evaluates $\phi_{ij}$ for
every line $j$ and every gas element $i$ (for example the cells crossed by a skewer), with:

- (a) five temperatures at rest: a hotter gas broadens the line;
- (b) five velocities at $10^4$ K: the photons travel along $+x$, so the elements with $v_x > 0$, moving with them
  (towards the observer), absorb at negative $v_\dagger$.

In [ ]:
v = jnp.linspace(-400, 400, 801)
fig, axs = plt.subplots(1, 2, figsize=(20, 6), layout="constrained")
for ax, (Tlo, Thi, vlo, vhi, title) in zip(axs, ((1e3, 1e6, 0.0, 0.0, r"$T$ from $10^3$ to $10^6$ K"), (1e4, 1e4, -300.0, 300.0, r"$v_x$ from −300 to +300 km/s, $T = 10^4$ K"))):
    T5 = np.geomspace(Tlo, Thi, 5)                                       # five gas elements
    u5 = np.linspace(vlo, vhi, 5)                                        # their v_x: the velocity along the photon (+x)
    phi = np.asarray(line_profiles(v, LineBundle(["HI-LyA"]), temperature=T5, los_velocity=u5, scale_factor=1.0, a_dagger=1.0))[0]
    for i in range(5):
        ax.semilogy(v, phi[i], color=list(TOL.values())[i])
    ax.set(xlabel=r"$v_\dagger$ [km/s]", ylabel=r"$\phi$ [ns]", ylim=(1e-8, None), title=title)
add_panel_letters(axs); fig.suptitle(r"Ly$\alpha$ profiles: temperature sets the width, the line-of-sight velocity the centre", fontsize=22); plt.show()